# 06. Provider Registry — lookup, validation, caching

`load_model()` is the front door to ArcLLM, but the *registry* is the
machinery behind that door. It does four jobs that the public API
deliberately hides:

1. **Provider name validation** — every name is matched against a
   strict regex *before* any module import, blocking path traversal
   and arbitrary code execution (OWASP ASI-04, NIST SI-10).
2. **Lazy adapter loading** — 17 provider adapters ship with arcllm,
   but `import arcllm` doesn't import a single one of them. Adapters
   are resolved on first use via `importlib.import_module` and a
   naming convention.
3. **Caching** — provider configs, adapter classes, the global config,
   and the vault resolver are all module-level dicts keyed off the
   provider name. Repeated `load_model("anthropic")` calls do not
   re-parse TOML, do not re-import modules, and do not re-build
   resolvers.
4. **Test isolation** — `clear_cache()` resets every cache
   *plus* downstream module state (rate-limit buckets, OTel SDK,
   telemetry budgets, telemetry global defaults). Calling it between
   notebook sections is the only reliable way to start from a clean
   baseline.

This notebook focuses on **registry mechanics**. If you want a deep
dive on every kwarg `load_model()` accepts, the module stack order,
or `on_event`/`trace_store`/`agent_label`/`routing` plumbing, read
`04-agentic-loop.ipynb` first — that material is *not* repeated here.

You will learn:

- The exact regex used to validate provider names, what it accepts,
  and what it rejects (with adversarial examples).
- How `_get_adapter_class()` turns a string like `"anthropic"` into
  the `AnthropicAdapter` class without a single hand-written mapping.
- How config and adapter caching behave under repeated calls and
  multi-provider loads.
- What `clear_cache()` actually clears (it's more than the registry).
- How the `routing=` kwarg integrates with the registry — without
  duplicating the full RoutingModule deep-dive in
  `17-routing-module.ipynb`.
- Common errors you will hit, what they look like, and how to fix
  each one.

Mock-first: every cell runs without an API key. The single live cell
(if any) is gated by `ANTHROPIC_API_KEY` — set it to opt in.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")


Pull in just what we need from `arcllm` for registry-focused work:
`load_model`, `clear_cache`, the `LLMProvider` ABC (the contract
adapters and the registry honor), and `ArcLLMConfigError` for the
intentional `# this raises` cells.

In [ ]:
from unittest.mock import AsyncMock, MagicMock, patch

import arcllm
from arcllm import (
    ArcLLMConfigError,
    LLMProvider,
    LLMResponse,
    Message,
    Usage,
    clear_cache,
    load_model,
    load_provider_config,
)

# Registry internals — exposed here for didactic walkthrough only.
# Application code should treat the registry as opaque.
from arcllm import registry as _registry
from arcllm.registry import (
    _VALID_PROVIDER_RE,
    _build_adapter,
    _get_adapter_class,
    _validate_provider_name,
)

print("arcllm version:", arcllm.__version__)
print("regex:", _VALID_PROVIDER_RE.pattern)


Set test API keys so the adapter constructors don't fail later
when we exercise the full `load_model()` path. The mocks short-circuit
real HTTP — these values are never sent anywhere.

In [ ]:
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key-not-real")
os.environ.setdefault("OPENAI_API_KEY", "test-key-not-real")

clear_cache()
print("API keys set; caches cleared.")


## 2. The registry's role — what it does that `load_model()` hides

`load_model()` is a thin orchestration shell. The interesting work
lives in `arcllm/registry.py`, which is roughly four pieces:

```
┌──────────────────────────────────────────────────────────┐
│  load_model(provider, model, **kwargs)                   │
│    │                                                     │
│    ├─ load_provider_config(provider)   ← parses TOML     │
│    │    └─ _validate_provider_name()   ← ASI-04 gate     │
│    │                                                     │
│    ├─ _get_adapter_class(provider)     ← importlib +     │
│    │    └─ _validate_provider_name()      naming convo   │
│    │                                                     │
│    ├─ _build_adapter(...)              ← resolves vault, │
│    │                                      constructs     │
│    │                                                     │
│    └─ _resolve_module_config(...) for each module        │
│         (sets up Otel, Queue, Telemetry, … wrappers)     │
└──────────────────────────────────────────────────────────┘
```

Five module-level caches sit alongside this flow:

| Cache | Keys | Purpose |
|---|---|---|
| `_provider_config_cache` | provider name | Parsed `ProviderConfig` from TOML |
| `_adapter_class_cache` | provider name | Adapter class object |
| `_global_config_cache` | (singleton) | Parsed `GlobalConfig` from `config.toml` |
| `_module_settings_cache` | module name | Dict of module settings (sans `enabled`) |
| `_vault_resolver_cache` | (singleton) | Vault-backed key resolver |

Every cache miss is followed by a `threading.Lock`-protected double-check
write. That makes the registry **thread-safe without the GIL** — important
because Arc must run cleanly under PEP 703 free-threading.

Confirm by inspection that those caches start empty after
`clear_cache()`.

In [ ]:
clear_cache()

print("provider_config_cache:", dict(_registry._provider_config_cache))
print("adapter_class_cache:  ", dict(_registry._adapter_class_cache))
print("global_config_cache:  ", _registry._global_config_cache)
print("module_settings_cache:", dict(_registry._module_settings_cache))
print("vault_resolver_cache: ", _registry._vault_resolver_cache)


One `load_model()` call populates all five (vault stays `None`
unless a vault backend is configured).

In [ ]:
clear_cache()
_ = load_model("anthropic")

print("provider_config_cache keys:", list(_registry._provider_config_cache.keys()))
print("adapter_class_cache keys:  ", list(_registry._adapter_class_cache.keys()))
print("global_config_cache type:  ", type(_registry._global_config_cache).__name__)
print("module_settings_cache keys:", sorted(_registry._module_settings_cache.keys()))
print("vault_resolver_cache:      ", _registry._vault_resolver_cache)


## 3. Provider name validation (ASI-04)

The registry treats provider names as **untrusted input**. The flow
is:

```
provider_name (string) ─► _validate_provider_name()
                              │
                              └─► importlib.import_module("arcllm.adapters." + name)
```

If validation is permissive, a hostile provider name turns into a
hostile import — the literal "agentic supply chain" attack from
**OWASP ASI-04**. Arc's defense is a strict allowlist regex:

```python
_VALID_PROVIDER_RE = re.compile(r"^[a-z][a-z0-9_]{0,63}$")
```

Reading that regex left-to-right:

| Position | Allowed | Why |
|---|---|---|
| First char | `[a-z]` (one lowercase letter) | Forbids `.`, `/`, `_`, digits — kills `.hidden`, `/etc/passwd`, `_secret`, `1bad` |
| Chars 2-64 | `[a-z0-9_]` (zero to 63 of) | No uppercase, no hyphens, no dots, no slashes |
| Total length | 1 to 64 chars | `{0,63}` after the leading char ⇒ 1-64 inclusive |
| End | `$` (anchored) | Stops trailing-dot or trailing-slash bypass |

This regex is enforced *before* any filesystem or import operation.
Mapped controls: **OWASP ASI-04**, **NIST 800-53 SI-10** (Information
Input Validation).

Show that valid names — including every adapter that ships with
arcllm — pass without complaint.

In [ ]:
valid_names = [
    "anthropic",
    "openai",
    "azure_openai",        # underscore
    "huggingface_tgi",     # multiple underscores
    "deepseek",
    "google",
    "cohere",
    "xai",
    "groq",
    "mistral",
    "moonshot",
    "fireworks",
    "together",
    "ollama",
    "vllm",
    "huggingface",
    "o1",                  # short + digit
    "my_custom_provider1", # mixed
]

for name in valid_names:
    _validate_provider_name(name)  # raises if invalid
print(f"all {len(valid_names)} valid names accepted")


Now exercise rejection. Each call below intentionally raises
`ArcLLMConfigError` — we catch it and print the message so you can
see the regex working.

In [ ]:
# this raises — every one of these is a textbook ASI-04 attempt
hostile = [
    "../etc/passwd",       # path traversal
    "..",                  # parent dir
    "../../secrets",       # deep path traversal
    ".hidden",             # leading dot
    "/absolute/path",      # leading slash
    "anthropic/openai",    # embedded slash
    "anthropic.openai",    # embedded dot (treats as submodule)
    "anthropic ",          # trailing whitespace
    " anthropic",          # leading whitespace
    "ANTHROPIC",           # uppercase
    "Anthropic",           # mixed case
    "anthropic-v2",        # hyphen — Python module names can't have these
    "1anthropic",          # leading digit
    "_anthropic",          # leading underscore (valid Python ident, but reserved)
    "os.system",           # builtin module probe
    "builtins",            # builtin module probe (passes regex; fails on import)
    "anthropic\x00",      # null byte
    "anthropic;rm -rf /",  # shell injection attempt
    "x" * 65,              # 65 chars — over the 64-char cap
]

rejected = 0
for name in hostile:
    try:
        _validate_provider_name(name)
    except ArcLLMConfigError:
        rejected += 1
print(f"rejected {rejected}/{len(hostile)} hostile names")


Note `"builtins"` actually passes the regex — it matches
`^[a-z][a-z0-9_]{0,63}$`. The defense in depth here is that
`importlib.import_module("arcllm.adapters.builtins")` then fails because
no such module exists, raising `ArcLLMConfigError("No adapter module
found for provider 'builtins'.")`. **Validation is the first gate, not
the last.**

In [ ]:
# this raises — proves the second gate (missing adapter module)
clear_cache()
try:
    load_model("builtins")
except ArcLLMConfigError as e:
    print("caught:", e)


A subtle one: NFKC-normalizing characters (full-width Latin,
mathematical alphanumeric symbols) look like ASCII to a human but are
distinct bytes. The regex is byte-strict — it never NFKC-normalizes
input — so spoofs are rejected.

For comparison, the `RoutingModule` *does* NFKC-validate
classifications before lookup (see `17-routing-module.ipynb`). The
provider regex is stricter because the consequence of bypass is
arbitrary `import` rather than a routing miss.

In [ ]:
# this raises — homoglyph and zero-width spoofs of "anthropic"
# Built with explicit \u escapes so the source stays plain ASCII.
spoofs = [
    "\uff41" + "nthropic",       # full-width latin small 'a' + 'nthropic'
    "anthr" + "\u043e" + "pic",  # cyrillic small 'o' homoglyph in the middle
    "anthropic" + "\u200b",      # trailing zero-width space
]

for spoof in spoofs:
    try:
        _validate_provider_name(spoof)
    except ArcLLMConfigError:
        print(f"rejected: {spoof!r}")


One more: `_validate_provider_name()` is also called from
`config.py::load_provider_config()` (the TOML loader). The two
validators share the same intent but live in different files because
each module owns its own input boundary. Both must pass for a
provider name to make it through `load_model()`.

In [ ]:
# Confirm the two regex patterns agree on the safe core character set.
from arcllm.config import _PROVIDER_NAME_RE
from arcllm.registry import _VALID_PROVIDER_RE

print("config.py:  ", _PROVIDER_NAME_RE.pattern)
print("registry.py:", _VALID_PROVIDER_RE.pattern)

# Round-trip every valid name through both — they must agree.
for name in valid_names:
    assert _PROVIDER_NAME_RE.match(name), name
    assert _VALID_PROVIDER_RE.match(name), name
print("both regexes accept every shipped provider name")


## 4. Lazy adapter loading — 17 adapters, zero import cost

ArcLLM ships with 17 adapters (Anthropic, OpenAI, Azure_OpenAI, Google,
Cohere, xAI, DeepSeek, Mistral, Groq, Moonshot, Fireworks, Together,
HuggingFace, HuggingFace_TGI, Ollama, vLLM, plus the ABC). Each pulls
in `httpx` and provider-specific transformations. Importing them all
on `import arcllm` would blow the cold-start budget.

The fix is two-tier laziness:

1. `arcllm/__init__.py` defines `_LAZY_IMPORTS` and a
   `__getattr__` hook. `from arcllm import AnthropicAdapter` only
   imports `arcllm.adapters.anthropic` on first attribute access.
2. `_get_adapter_class()` uses `importlib.import_module` keyed on the
   provider name and a `{name.title()}Adapter` class lookup. There is
   no central mapping dict — the file structure *is* the registry.

The result: `import arcllm` touches `types.py`, `config.py`,
`exceptions.py`, `registry.py`, and a few sibling modules, but
**not a single adapter file**.

Show which adapter modules are imported after a fresh
`import arcllm`. Restart-style baseline — we use `sys.modules` to
inspect what's loaded right now.

In [ ]:
adapter_modules_loaded = sorted(
    name for name in sys.modules if name.startswith("arcllm.adapters.")
)
print("adapter modules in sys.modules:")
for name in adapter_modules_loaded:
    print(" ", name)


At least one adapter module is loaded already because the tests
or earlier cells touched `load_model("anthropic")`. The point of the
laziness isn't that *no* adapter is ever loaded — it's that
**adapters you don't use are never loaded**.

Demonstrate by counting `arcllm.adapters.*` modules before and after
loading three different providers.

In [ ]:
before = {n for n in sys.modules if n.startswith("arcllm.adapters.")}
clear_cache()
_ = load_model("anthropic")
_ = load_model("openai")
# Mock the third one so we don't need its API key in env.
os.environ.setdefault("MISTRAL_API_KEY", "test-key-not-real")
_ = load_model("mistral")

after = {n for n in sys.modules if n.startswith("arcllm.adapters.")}
print("loaded by these three calls:", sorted(after - before))
print("never imported:", sorted(
    f"arcllm.adapters.{p}"
    for p in [
        "google", "cohere", "xai", "groq", "moonshot",
        "fireworks", "together", "vllm", "ollama",
        "huggingface", "huggingface_tgi",
    ]
    if f"arcllm.adapters.{p}" not in after
))


Now look at the convention itself. `_get_adapter_class()`
implements *precisely* this:

```python
module_path = f"arcllm.adapters.{provider_name}"
module = importlib.import_module(module_path)
class_name = f"{provider_name.title()}Adapter"
adapter_class = getattr(module, class_name)
```

The `provider_name.title()` step is the reason that `OpenAIAdapter`
was renamed to `OpenaiAdapter` and `Azure_OpenaiAdapter` is spelled
the way it is — they have to round-trip through `str.title()`.

In [ ]:
# Verify the convention round-trips for every shipped adapter.
shipped = [
    "anthropic", "azure_openai", "cohere", "deepseek", "fireworks",
    "google", "groq", "huggingface", "huggingface_tgi", "mistral",
    "moonshot", "ollama", "openai", "together", "vllm", "xai",
]

for name in shipped:
    expected_class = f"{name.title()}Adapter"
    expected_module = f"arcllm.adapters.{name}"
    print(f"  {name:20s} -> {expected_module:35s} :: {expected_class}")


Since the cache is keyed on provider name, repeated calls hit
the cache without re-running `importlib.import_module`.

In [ ]:
clear_cache()
real_import = __import__("importlib").import_module

with patch("importlib.import_module", wraps=real_import) as mock_import:
    cls1 = _get_adapter_class("anthropic")
    cls2 = _get_adapter_class("anthropic")
    cls3 = _get_adapter_class("anthropic")

print("import_module calls for 3x same provider:", mock_import.call_count)
print("classes identical:", cls1 is cls2 is cls3)


## 5. Caching behavior

The registry has two cache shapes:

- **Provider-keyed**: `_provider_config_cache`, `_adapter_class_cache`.
  Same provider string ⇒ same cached object.
- **Singleton**: `_global_config_cache`, `_vault_resolver_cache`.
  One per process. Loaded on first need.

Same `(provider, model)` arguments do **not** produce the same
adapter instance — `load_model()` always builds a fresh chain. The
caches make that *cheap* (no TOML re-parse, no import) but the
returned `LLMProvider` is freshly constructed every call. This
matters when you stack modules: each `load_model()` builds its own
queue, its own circuit breaker, its own retry counter.

Show that the underlying *config* object is shared (cache hit),
even though the *adapter* instance is fresh.

In [ ]:
clear_cache()
m1 = load_model("anthropic")
m2 = load_model("anthropic")

# Different adapter instance each call — module stack is fresh.
print("m1 is m2:", m1 is m2)
print("type(m1):", type(m1).__name__)
print("type(m2):", type(m2).__name__)

# But the cached ProviderConfig is shared — TOML parsed exactly once.
cfg1 = _registry._provider_config_cache["anthropic"]
cfg2 = _registry._provider_config_cache["anthropic"]
print("config object shared:", cfg1 is cfg2)


Prove the TOML parse runs exactly once across many calls by
counting wraps on `load_provider_config`.

In [ ]:
from arcllm.config import load_provider_config as real_loader

clear_cache()

with patch("arcllm.registry.load_provider_config", wraps=real_loader) as mock_load:
    for _ in range(10):
        load_model("anthropic")
print("load_provider_config calls for 10x same provider:", mock_load.call_count)


Two providers ⇒ two separate cache entries, two parse calls
total — never any cross-pollution.

In [ ]:
clear_cache()

with patch("arcllm.registry.load_provider_config", wraps=real_loader) as mock_load:
    for _ in range(5):
        load_model("anthropic")
    for _ in range(5):
        load_model("openai")
print("load_provider_config calls (5x anthropic + 5x openai):", mock_load.call_count)
print("cache keys:", sorted(_registry._provider_config_cache.keys()))


`clear_cache()` is the test-isolation hook. It does **more**
than dump the registry's caches — it also resets:

- `arcllm.modules.rate_limit.clear_buckets()` — token-bucket state
- `arcllm.modules.otel.reset_sdk()` — OTel SDK provider
- `arcllm.modules.telemetry.clear_budgets()` — budget accumulators
- `arcllm.modules.telemetry.clear_global_defaults()` — telemetry
  global defaults

If you're writing tests that touch any of those modules, calling
`clear_cache()` at the start of each test is the only reliable way
to start clean.

In [ ]:
# Demonstrate cache behavior across a clear.
clear_cache()

with patch("arcllm.registry.load_provider_config", wraps=real_loader) as mock_load:
    load_model("anthropic")  # 1: cold
    load_model("anthropic")  # 2: cache hit
    clear_cache()
    load_model("anthropic")  # 3: cold again (cache cleared)
    load_model("anthropic")  # 4: cache hit

print("load_provider_config calls:", mock_load.call_count)
print("expected: 2 (one cold load before clear, one after)")


## 6. Multi-provider scenarios — cache isolation

Real agents often need two providers concurrently: a primary for
heavy lifting plus a cheaper fallback for routine queries. Or one
classification routes to a Gov-cloud provider while another routes
to commercial. Whatever the topology, the registry keeps each
provider's state isolated.

The keys make this trivially true:

- `_provider_config_cache` is keyed on provider name.
- `_adapter_class_cache` is keyed on provider name.
- Each call to `load_model("X")` builds a fresh adapter for X using
  X's cached config and class.

There is no shared mutable state between providers. Adding a third
provider does not invalidate the first two.

In [ ]:
clear_cache()

# Build three independent adapters.
anth = load_model("anthropic")
oai = load_model("openai")
mistral = load_model("mistral")

print("cache keys:", sorted(_registry._provider_config_cache.keys()))
print("anthropic class:", type(anth).__name__)
print("openai class:   ", type(oai).__name__)
print("mistral class:  ", type(mistral).__name__)

# Disable modules to see the bare adapters underneath.
print()
print("with all modules off:")
for name in ("anthropic", "openai", "mistral"):
    bare = load_model(
        name,
        telemetry=False, retry=False, queue=False, security=False,
        audit=False, rate_limit=False, fallback=False,
        circuit_breaker=False, otel=False,
    )
    print(f"  {name:10s} -> {type(bare).__name__}")


Now show the inverse: clearing the cache on a fresh
`load_model("openai")` does not affect a previously-built Anthropic
adapter. The adapter holds its own config reference; the cache is
just a shortcut for *future* lookups.

In [ ]:
clear_cache()
anth_before = load_model("anthropic")
anth_cfg_id_before = id(anth_before._config) if hasattr(anth_before, "_config") else None

clear_cache()  # nuke all caches
oai_after = load_model("openai")  # rebuild a different provider

# anth_before is still alive and still functional — it captured its config
# at construction time. The cache exists for the *next* load_model() call.
print("anthropic adapter still has _config:", hasattr(anth_before, "_config"))
print("anthropic cache key after clear+openai:",
      sorted(_registry._provider_config_cache.keys()))


## 7. The `routing=` kwarg — integration overview

When `routing=` is set with a non-empty `rules` dict, `load_model()`
swaps the single adapter at the innermost stack position for a
`RoutingModule` that holds **one adapter per classification**. Each
adapter is built through `_build_adapter()` — meaning each routing
rule goes through the same name-validation, config-cache, and
adapter-class-cache flow as a top-level `load_model()` call.

The contract is intentionally simple at the registry layer:

```python
load_model(
    "anthropic",  # ignored when routing rules cover everything
    routing={
        "rules": {
            "unclassified": {"provider": "openai"},
            "cui":          {"provider": "anthropic"},
        },
        "default_classification": "unclassified",
        "enforcement": "block",   # or "warn"
    },
)
```

Validation, NFKC normalization on classifications, enforcement
modes, and per-call dispatch live in `RoutingModule` itself —
**deep dive in `17-routing-module.ipynb`**. Here we only verify
that the registry threads the kwarg through correctly.

In [ ]:
from arcllm.modules.routing import RoutingModule

clear_cache()


def fake_build(name, model, *_a, **_kw):
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = model or f"{name}-default"
    inner.invoke = AsyncMock(
        return_value=LLMResponse(
            content=f"hello from {name}",
            usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
            model=inner.model_name,
            stop_reason="end_turn",
        )
    )
    return inner


with patch("arcllm.registry._build_adapter", side_effect=fake_build):
    routed = load_model(
        "anthropic",
        telemetry=False, retry=False, queue=False, security=False,
        audit=False, rate_limit=False, fallback=False,
        circuit_breaker=False, otel=False,
        routing={
            "rules": {
                "unclassified": {"provider": "openai"},
                "cui":          {"provider": "anthropic"},
            },
            "default_classification": "unclassified",
            "enforcement": "block",
        },
    )

print("type:", type(routed).__name__)
print("isinstance RoutingModule:", isinstance(routed, RoutingModule))
print("known classifications:", sorted(routed._adapters.keys()))
print("default classification:", routed._default_classification)
print("enforcement:", routed._enforcement)


Each rule went through `_build_adapter()`, which means each
provider name was validated. Try a routing rule with an invalid
provider name — it raises during construction.

In [ ]:
# this raises — bad provider name in a routing rule
clear_cache()

try:
    load_model(
        "anthropic",
        routing={
            "rules": {
                "unclassified": {"provider": "../etc/passwd"},
            },
            "default_classification": "unclassified",
        },
    )
except ArcLLMConfigError as e:
    print("caught:", e)


Routing rules with no `provider` key raise too — the registry
checks for it explicitly because the alternative (silently skipping
or defaulting) would be a routing security hole.

In [ ]:
# this raises — routing rule missing the 'provider' key
clear_cache()

try:
    load_model(
        "anthropic",
        routing={
            "rules": {
                "unclassified": {},  # no provider!
            },
            "default_classification": "unclassified",
        },
    )
except ArcLLMConfigError as e:
    print("caught:", e)


Per-call dispatch (`classification="cui"` in `invoke()`),
NFKC validation, and the difference between `enforcement="warn"` and
`enforcement="block"` are out of scope here. Read
`17-routing-module.ipynb` for the full contract.

## 8. Common errors — what they look like, how to fix them

Every registry error raises `ArcLLMConfigError` with a message that
names the failure mode. Memorize this table:

| Symptom | Cause | Fix |
|---|---|---|
| `Invalid provider name '<x>'.` | Name fails the `[a-z][a-z0-9_]{0,63}` regex | Use lowercase ASCII; no hyphens, dots, slashes |
| `Provider name cannot be empty` | Empty string passed to `load_model("")` | Pass a real provider name |
| `Provider name too long (max 64 characters)` | >64 char string | Shorten the name |
| `Provider config not found: 'X'` | No `providers/X.toml` shipped | Add a TOML; check spelling |
| `No adapter module found for provider 'X'.` | TOML present, no `adapters/X.py` | Add the adapter module |
| `No adapter class 'XAdapter' found in module 'arcllm.adapters.X'` | Module present, class name doesn't match `{X.title()}Adapter` | Rename the class |
| `Routing rule 'Y' missing 'provider'` | A routing rule had no `provider` key | Add `"provider": "..."` |
| Adapter `__init__` raises | API key not set in env | `export PROVIDER_API_KEY=...` |

Empty string — caught early at the config layer.

In [ ]:
# this raises — empty provider name
clear_cache()
try:
    load_model("")
except ArcLLMConfigError as e:
    print("caught:", e)


Missing TOML — name is valid, but no `providers/<name>.toml`
exists in the package.

In [ ]:
# this raises — valid name, no TOML
clear_cache()
try:
    load_model("nonexistent_provider_xyz")
except ArcLLMConfigError as e:
    print("caught:", e)


Missing adapter module — the TOML loads but no Python module
matches the convention. Simulated here with a `patch` because we
don't have an orphan TOML to ship.

In [ ]:
# this raises — TOML lookup succeeds, adapter module is missing
clear_cache()

with patch("arcllm.registry.load_provider_config") as mock_cfg:
    fake_provider = type("FakeProvider", (), {"default_model": "test"})()
    fake_config = type("FakeConfig", (), {"provider": fake_provider})()
    mock_cfg.return_value = fake_config

    try:
        load_model("nosuchadapter")
    except ArcLLMConfigError as e:
        print("caught:", e)


Missing adapter *class* — the module exists but the
class name doesn't match the convention. Same simulated path.

In [ ]:
# this raises — module loads, but no XAdapter class inside
import types as stdlib_types

clear_cache()

with patch("importlib.import_module") as mock_import:
    fake_module = stdlib_types.ModuleType("arcllm.adapters.fakeprov")
    # Note: no FakeprovAdapter attribute defined.
    mock_import.return_value = fake_module
    try:
        _get_adapter_class("fakeprov")
    except ArcLLMConfigError as e:
        print("caught:", e)


Missing API key — caught at adapter construction time.
Provider TOML and adapter class are both fine; the *credential*
isn't reachable.

In [ ]:
# this raises — adapter constructor fails because the API key env var is unset
clear_cache()

saved = os.environ.pop("ANTHROPIC_API_KEY", None)
try:
    load_model("anthropic")
except ArcLLMConfigError as e:
    print("caught:", e)
finally:
    if saved is not None:
        os.environ["ANTHROPIC_API_KEY"] = saved
    else:
        os.environ["ANTHROPIC_API_KEY"] = "test-key-not-real"


The pattern is consistent: every failure mode raises
`ArcLLMConfigError` with a message that names the file or symbol
that needs attention. Catch broadly, then route to UX based on the
message string if you must.

## 9. Pointer to the `load_model` deep dive

The registry's job ends at "I built you an adapter chain". Everything
*about* that chain — kwarg semantics, default module stack, the
`on_event`/`trace_store`/`agent_label` plumbing, when each module
fires, debugging via `_inner` walks — lives in
**`arcllm/04-agentic-loop.ipynb`**.

Quick map of where to read what:

| Question | Notebook |
|---|---|
| What does `load_model()` accept? | `04-agentic-loop.ipynb` |
| In what order are modules wrapped? | `04-agentic-loop.ipynb` § 12 |
| How does `on_event` get into TelemetryModule? | `04-agentic-loop.ipynb` § 5 |
| `trace_store` API and chain verification | `14-trace-store.ipynb` |
| `RoutingModule` semantics, NFKC, enforcement | `17-routing-module.ipynb` |
| `BaseModule` contract, custom modules | `07-module-system.ipynb` |
| `QueueModule`, `CircuitBreakerModule` | `15-queue-circuit-breaker.ipynb` |

Do one quick read-through of `04-agentic-loop.ipynb` after this one
to see how registry caching, validation, and lazy loading slot into
the bigger picture.

One small parting trick: every `BaseModule` in the chain
exposes a `_inner` attribute. Walk it to see what `load_model()`
gave you.

In [ ]:
def stack_layers(provider) -> list[str]:
    layers = []
    cur = provider
    while True:
        layers.append(type(cur).__name__)
        inner = getattr(cur, "_inner", None)
        if inner is None:
            break
        cur = inner
    return layers


clear_cache()
default = load_model("anthropic")
bare = load_model(
    "anthropic",
    telemetry=False, retry=False, queue=False, security=False,
    audit=False, rate_limit=False, fallback=False,
    circuit_breaker=False, otel=False,
)
print("default stack:", " -> ".join(stack_layers(default)))
print("bare stack:   ", " -> ".join(stack_layers(bare)))


## 10. Summary

The registry is what sits between a string like `"anthropic"` and a
fully-wired LLM call. Its surface is small (`load_model`,
`clear_cache`) but the mechanics behind that surface enforce hard
security and performance properties:

- **ASI-04 input validation** — `_validate_provider_name()` rejects
  every form of path traversal, code injection, or homoglyph spoof
  before any `importlib.import_module()` call. Map: OWASP ASI-04,
  NIST SI-10.
- **Naming-convention discovery** — `_get_adapter_class()` resolves
  `provider_name` ⇒ module path + class name with no hand-written
  mapping. The 17 shipped adapters are reachable, plus any
  third-party adapter that follows the same convention.
- **Lazy loading at two tiers** — `arcllm/__init__.py::__getattr__`
  for top-level imports, `_get_adapter_class` for registry lookups.
  `import arcllm` does not import a single adapter module.
- **Five module-level caches** — provider config, adapter class,
  global config, module settings, vault resolver. Cache misses are
  lock-protected for free-threading safety.
- **`clear_cache()` does more than the registry** — it resets
  rate-limit buckets, OTel SDK, telemetry budgets, and global
  defaults. Required for clean test isolation.
- **Multi-provider isolation** — same code path for one or 17
  providers; no shared mutable state.
- **`routing=` integration** — registry threads each rule through
  `_build_adapter()` so every routing target gets the same
  validation and caching.
- **Errors are `ArcLLMConfigError`** — every failure mode names the
  file or symbol that needs attention; one catch type covers the
  whole surface.

Where to go next:

- `04-agentic-loop.ipynb` — every kwarg of `load_model()`, the full
  module stack, `on_event` and `trace_store` plumbing.
- `07-module-system.ipynb` — `BaseModule` contract, custom modules.
- `15-queue-circuit-breaker.ipynb` — Queue and CircuitBreaker
  end-to-end.
- `17-routing-module.ipynb` — RoutingModule deep dive.
- `02-config-loading.ipynb` — provider TOML format and layered
  config precedence.

Public API exercised in this notebook:

`arcllm.load_model`, `arcllm.clear_cache`, `arcllm.load_provider_config`,
`arcllm.LLMProvider`, `arcllm.LLMResponse`, `arcllm.Message`,
`arcllm.Usage`, `arcllm.ArcLLMConfigError`,
`arcllm.modules.routing.RoutingModule`,
plus the registry internals
`_VALID_PROVIDER_RE`, `_validate_provider_name`, `_get_adapter_class`,
and `_build_adapter` (shown for didactic purposes; treat as opaque
in production code).